# Content Effectiveness Analyzer

## Objetivos de Aprendizaje

En este notebook, aprenderás a learn how to:

1. **Analyze content effectiveness** using `CORTEX.COMPLETE()`
2. **Score content quality** with LLM prompts
3. **Extract key messages** from marketing materials
4. **Compare content versions** (A/B testing)
5. **Generar optimization recommendations** with AI

---

## What You'll Build

A **content analysis system** that scores marketing effectiveness:
- LLM-powered content scoring
- Key message extraction
- A/B test winner determination
- Readability analysis
- Content optimization recommendations

---

## Technical Requirements

- **Snowflake account** with standard SQL access
- **Approximately 12 minutes** to complete
- **100% SQL** (except Streamlit dashboard)

---

## Key Snowflake Features Used

- `CORTEX.COMPLETE()` - LLM for content analysis
- Prompt engineering - Structured LLM instructions
- `LENGTH()` and `REGEXP_COUNT()` - Text metrics
- `SPLIT_PART()` - Parse structured LLM responses
- Content versioning - Track changes over time

Let's begin!

---

## Paso 1: Configuración del Entorno

### Content Effectiveness Analysis

Measure and optimize:
- **Email campaigns**: Subject lines, body content, CTAs
- **Web pages**: Product detail pages, educational content
- **HCP materials**: Detailing aids, clinical studies

### Why Content Analysis Matters

- Identify high-performing content patterns
- A/B test variations objectively
- Optimize messaging for target audiences
- Improve ROI on content production

In [ ]:
CREATE DATABASE IF NOT EXISTS CONTENT_DB;
CREATE SCHEMA IF NOT EXISTS CONTENT_DB.PUBLIC;
USE SCHEMA CONTENT_DB.PUBLIC;

CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH 
    WITH WAREHOUSE_SIZE = 'SMALL' AUTO_SUSPEND = 60 AUTO_RESUME = TRUE;

USE WAREHOUSE COMPUTE_WH;

SELECT CURRENT_DATABASE() as db, CURRENT_SCHEMA() as schema, CURRENT_WAREHOUSE() as wh;

---

## Paso 2: Define Data Structure

### Tables

1. **`content_library`** - All marketing materials
2. **`content_performance`** - Engagement metrics per content
3. **`ab_tests`** - A/B test results

### Engagement Metrics

- **Impressions**: How many saw it
- **Opens/Views**: Engagement rate
- **Clicks**: CTA effectiveness
- **Conversions**: Ultimate goal (Rx, sign-up, download)

In [ ]:
-- Content library
CREATE OR REPLACE TABLE content_library (
    content_id VARCHAR(50) PRIMARY KEY,
    content_type VARCHAR(50),  -- Email, Webpage, HCP_Material
    content_title VARCHAR(200),
    content_text VARCHAR(5000),
    target_audience VARCHAR(50),  -- HCP, Patient, Payer
    created_date DATE,
    ab_test_variant VARCHAR(10)  -- A, B, Control
);

-- Performance metrics
CREATE OR REPLACE TABLE content_performance (
    performance_id VARCHAR(50) PRIMARY KEY,
    content_id VARCHAR(50),
    impressions INTEGER,
    opens_or_views INTEGER,
    clicks INTEGER,
    conversions INTEGER,
    measurement_date DATE
);

SELECT 'Tables created!' as status;

---

## Paso 3: Generar Datos Sintéticos Content Data

Creating realistic marketing content with variations for A/B testing

In [ ]:
-- Generar content library
INSERT INTO content_library
WITH content_templates AS (
    SELECT * FROM (VALUES
        ('Email', 'HCP', 'A', 'New Clinical Data: Xarelto Shows Superior Stroke Prevention', 
         'Recent study demonstrates Xarelto reduces stroke risk by 35% vs. 25% for warfarin in AF patients. View full results and patient profiles.'),
        ('Email', 'HCP', 'B', 'Achieve Better Outcomes with Xarelto', 
         'Help your AF patients reduce stroke risk. Xarelto: proven efficacy, once-daily convenience, no routine monitoring. Learn more about dosing.'),
        ('Email', 'Patient', 'A', 'Protect Against Stroke with Daily Xarelto', 
         'Just one tablet per day. Real patients see reduced stroke risk and better quality of life. Talk to your doctor today.'),
        ('Email', 'Patient', 'B', 'Real Results: John Avoided Stroke with Xarelto', 
         'Read John story of managing his AFib with daily Xarelto. No stroke events in 3 years. Could it work for you?'),
        ('Webpage', 'HCP', 'Control', 'Xarelto Prescribing Information', 
         'Complete prescribing information, dosing guidelines, and clinical trial data for Xarelto in atrial fibrillation management.'),
        ('HCP_Material', 'HCP', 'A', 'Xarelto Efficacy Overview for Cardiologists', 
         'Meta-analysis of 8 clinical trials shows consistent stroke risk reductions of 30-40% across patient populations. Bleeding risk profile demonstrated in ROCKET-AF.')
    ) t(content_type, target_audience, ab_variant, title, text)
)
SELECT 
    'CONTENT' || LPAD(SEQ4(), 5, '0') as content_id,
    ct.content_type,
    ct.title as content_title,
    ct.text as content_text,
    ct.target_audience,
    DATEADD('day', -FLOOR(UNIFORM(30, 180, RANDOM())), CURRENT_DATE()) as created_date,
    ct.ab_variant as ab_test_variant
FROM content_templates ct,
TABLE(GENERATOR(ROWCOUNT => 200))
WHERE UNIFORM(0, 1, RANDOM()) > (SEQ4() / 220.0);

-- Generar performance data
INSERT INTO content_performance
SELECT 
    UUID_STRING() as performance_id,
    c.content_id,
    -- Impressions vary by channel
    CASE c.content_type
        WHEN 'Email' THEN FLOOR(UNIFORM(500, 5000, RANDOM()))
        WHEN 'Webpage' THEN FLOOR(UNIFORM(1000, 10000, RANDOM()))
        ELSE FLOOR(UNIFORM(200, 2000, RANDOM()))
    END as impressions,
    -- Open/view rate varies by variant quality
    FLOOR(impressions * CASE c.ab_test_variant
        WHEN 'A' THEN UNIFORM(0.15, 0.35, RANDOM())
        WHEN 'B' THEN UNIFORM(0.20, 0.40, RANDOM())
        ELSE UNIFORM(0.18, 0.30, RANDOM())
    END) as opens_or_views,
    -- Click rate
    FLOOR(opens_or_views * UNIFORM(0.05, 0.25, RANDOM())) as clicks,
    -- Conversion rate
    FLOOR(clicks * UNIFORM(0.02, 0.15, RANDOM())) as conversions,
    c.created_date as measurement_date
FROM content_library c;

-- Verify data
SELECT 
    (SELECT COUNT(*) FROM content_library) as content_count,
    (SELECT COUNT(*) FROM content_performance) as performance_records;

---

## Paso 4: Score Content Quality with Cortex AI

### Using COMPLETE() for Content Analysis

Ask the LLM para puntuar content on:
- **Clarity**: Is the message clear?
- **Relevance**: Does it address audience needs?
- **Call-to-action**: Is the CTA compelling?
- **Tone**: Appropriate for audience?

**Returns**: Structured scores 1-10

In [ ]:
-- Score content using Cortex COMPLETE
CREATE OR REPLACE TABLE content_quality_scores AS
SELECT 
    c.content_id,
    c.content_type,
    c.content_title,
    c.target_audience,
    c.ab_test_variant,
    -- Use Cortex para puntuar content quality
    AI_COMPLETE(
        'mistral-large2',
        CONCAT(
            'You are a pharmaceutical marketing expert. Score this content on a scale of 1-10 for: ',
            'Clarity, Relevance to ',c.target_audience,', Call-to-Action strength. ',
            'Return ONLY three numbers separated by commas (e.g., "8,7,9"). ',
            'Content: "',c.content_title,' - ',SUBSTRING(c.content_text, 1, 200),'"'
        )
    ) as ai_scores_raw,
    -- Parse scores (format: "8,7,9")
    TRY_CAST(SPLIT_PART(ai_scores_raw, ',', 1) AS INTEGER) as clarity_score,
    TRY_CAST(SPLIT_PART(ai_scores_raw, ',', 2) AS INTEGER) as relevance_score,
    TRY_CAST(SPLIT_PART(ai_scores_raw, ',', 3) AS INTEGER) as cta_score,
    -- Composite quality score
    (COALESCE(clarity_score, 5) + COALESCE(relevance_score, 5) + COALESCE(cta_score, 5)) / 3.0 as avg_quality_score
FROM content_library c
LIMIT 50;  -- Limit for cost control

-- View top-scoring content
SELECT 
    content_title,
    target_audience,
    ab_test_variant,
    clarity_score,
    relevance_score,
    cta_score,
    ROUND(avg_quality_score, 1) as quality_score
FROM content_quality_scores
WHERE clarity_score IS NOT NULL
ORDER BY avg_quality_score DESC
LIMIT 15;

---

## Paso 5: Calculate Engagement Metrics

### Funnel Analysis

Track drop-off at each stage:
1. **Impressions** → 2. **Opens/Views** → 3. **Clicks** → 4. **Conversions**

### Key Rates

- **Open rate** = Opens / Impressions
- **Click-through rate (CTR)** = Clicks / Opens
- **Conversion rate** = Conversions / Clicks

In [ ]:
-- Calculate engagement metrics
CREATE OR REPLACE TABLE engagement_metrics AS
SELECT 
    c.content_id,
    c.content_type,
    c.content_title,
    c.target_audience,
    c.ab_test_variant,
    p.impressions,
    p.opens_or_views,
    p.clicks,
    p.conversions,
    -- Calculate rates
    ROUND((p.opens_or_views::FLOAT / NULLIF(p.impressions, 0)) * 100, 2) as open_rate_pct,
    ROUND((p.clicks::FLOAT / NULLIF(p.opens_or_views, 0)) * 100, 2) as ctr_pct,
    ROUND((p.conversions::FLOAT / NULLIF(p.clicks, 0)) * 100, 2) as conversion_rate_pct,
    -- Overall effectiveness
    ROUND((p.conversions::FLOAT / NULLIF(p.impressions, 0)) * 100, 3) as overall_conversion_pct,
    -- Quality scores (if available)
    cqs.avg_quality_score
FROM content_library c
JOIN content_performance p ON c.content_id = p.content_id
LEFT JOIN content_quality_scores cqs ON c.content_id = cqs.content_id;

-- Top performing content
SELECT 
    content_title,
    content_type,
    target_audience,
    impressions,
    open_rate_pct || '%' as open_rate,
    ctr_pct || '%' as ctr,
    conversion_rate_pct || '%' as conv_rate,
    conversions
FROM engagement_metrics
WHERE open_rate_pct > 0
ORDER BY overall_conversion_pct DESC
LIMIT 15;

---

## Paso 6: A/B Test Analysis

### Statistical Comparison

Compare variant A vs. B vs. Control:
- Which has higher conversion rate?
- Is the difference statistically significant?
- Calculate confidence intervals

In [ ]:
-- A/B test results analysis
CREATE OR REPLACE TABLE ab_test_results AS
WITH variant_performance AS (
    SELECT 
        ab_test_variant,
        content_type,
        target_audience,
        COUNT(DISTINCT content_id) as content_count,
        SUM(impressions) as total_impressions,
        SUM(opens_or_views) as total_opens,
        SUM(clicks) as total_clicks,
        SUM(conversions) as total_conversions,
        ROUND(AVG(open_rate_pct), 2) as avg_open_rate,
        ROUND(AVG(ctr_pct), 2) as avg_ctr,
        ROUND(AVG(conversion_rate_pct), 2) as avg_conversion_rate,
        ROUND(AVG(COALESCE(avg_quality_score, 5)), 1) as avg_ai_quality_score
    FROM engagement_metrics
    GROUP BY ab_test_variant, content_type, target_audience
),
control_baseline AS (
    SELECT 
        content_type,
        target_audience,
        avg_conversion_rate as control_conversion_rate
    FROM variant_performance
    WHERE ab_test_variant = 'Control'
)
SELECT 
    vp.*,
    cb.control_conversion_rate,
    -- Lift vs. control
    ROUND(((vp.avg_conversion_rate - COALESCE(cb.control_conversion_rate, vp.avg_conversion_rate)) 
           / NULLIF(cb.control_conversion_rate, 0)) * 100, 1) as lift_vs_control_pct,
    -- Winner determination
    CASE 
        WHEN vp.avg_conversion_rate > COALESCE(cb.control_conversion_rate, 0) * 1.1 THEN '🏆 Winner'
        WHEN vp.avg_conversion_rate > COALESCE(cb.control_conversion_rate, 0) * 1.05 THEN '✅ Promising'
        ELSE '➡️ Similar'
    END as performance_vs_control
FROM variant_performance vp
LEFT JOIN control_baseline cb 
    ON vp.content_type = cb.content_type 
    AND vp.target_audience = cb.target_audience
ORDER BY vp.content_type, vp.target_audience, vp.avg_conversion_rate DESC;

-- View A/B test winners
SELECT 
    content_type,
    target_audience,
    ab_test_variant,
    content_count,
    total_conversions,
    avg_conversion_rate || '%' as conversion_rate,
    COALESCE(lift_vs_control_pct, 0) || '%' as lift,
    performance_vs_control as result
FROM ab_test_results
ORDER BY content_type, avg_conversion_rate DESC;

---

## Paso 7: Generar Content Recommendations

Combine AI quality scores with engagement data to recommend best practices

In [ ]:
-- Content optimization recommendations
CREATE OR REPLACE VIEW content_recommendations AS
WITH best_performers AS (
    SELECT 
        content_type,
        target_audience,
        content_title,
        avg_quality_score,
        overall_conversion_pct,
        open_rate_pct,
        ctr_pct,
        ROW_NUMBER() OVER (
            PARTITION BY content_type, target_audience 
            ORDER BY overall_conversion_pct DESC
        ) as performance_rank
    FROM engagement_metrics
    WHERE avg_quality_score IS NOT NULL
),
recommendations AS (
    SELECT 
        content_type,
        target_audience,
        'Top Performer Example' as recommendation_type,
        content_title as example_content,
        ROUND(avg_quality_score, 1) as quality_score,
        ROUND(overall_conversion_pct, 3) || '%' as conversion_rate,
        CONCAT(
            'This ',LOWER(content_type),' achieved ',
            ROUND(open_rate_pct,1),'% open rate and ',
            ROUND(ctr_pct,1),'% CTR. ',
            'Key success factors: High clarity (',quality_score,'/10) and strong CTA.'
        ) as insight
    FROM best_performers
    WHERE performance_rank <= 2
)
SELECT * FROM recommendations
ORDER BY content_type, target_audience;

-- View recommendations
SELECT 
    content_type,
    target_audience,
    example_content,
    quality_score,
    conversion_rate,
    insight
FROM content_recommendations
LIMIT 10;

---

## Paso 8: Interactive Content Performance Dashboard

Visualize A/B test results, top performers, and optimization insights

In [ ]:
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()

st.title("📄 Content Effectiveness Analyzer")
st.markdown("### AI-powered content optimization")

# Key metrics
total_content = session.sql("SELECT COUNT(*) as cnt FROM content_library").collect()[0]['CNT']
total_conversions = session.sql("SELECT SUM(conversions) as total FROM content_performance").collect()[0]['TOTAL']
avg_open_rate = session.sql("SELECT ROUND(AVG(open_rate_pct),1) as rate FROM engagement_metrics").collect()[0]['RATE']
best_variant = session.sql("""
    SELECT ab_test_variant 
    FROM ab_test_results 
    WHERE performance_vs_control = '🏆 Winner' 
    LIMIT 1
""").collect()

col1, col2, col3, col4 = st.columns(4)
with col1:
    st.metric("Total Content", int(total_content))
with col2:
    st.metric("Total Conversions", int(total_conversions))
with col3:
    st.metric("Avg Open Rate", f"{avg_open_rate}%")
with col4:
    winner = best_variant[0]['AB_TEST_VARIANT'] if len(best_variant) > 0 else 'TBD'
    st.metric("Best Variant", winner)

# A/B test results
st.markdown("---")
st.subheader("🏆 A/B Test Results")

ab_results = session.sql("""
    SELECT content_type, target_audience, ab_test_variant, 
           avg_conversion_rate, lift_vs_control_pct, performance_vs_control
    FROM ab_test_results
    ORDER BY content_type, avg_conversion_rate DESC
""").to_pandas()

st.dataframe(ab_results, use_container_width=True, hide_index=True)

# Top performers
st.markdown("---")
st.subheader("📈 Top Performing Content")

top_content = session.sql("""
    SELECT content_title, content_type, target_audience,
           open_rate_pct, ctr_pct, conversion_rate_pct, conversions
    FROM engagement_metrics
    WHERE open_rate_pct > 0
    ORDER BY overall_conversion_pct DESC
    LIMIT 10
""").to_pandas()

st.dataframe(top_content, use_container_width=True, hide_index=True)

# Quality scores vs. performance
st.markdown("---")
st.subheader("🎯 AI Quality Scores vs. Actual Performance")

if session.sql("SELECT COUNT(*) as cnt FROM content_quality_scores WHERE clarity_score IS NOT NULL").collect()[0]['CNT'] > 0:
    quality_perf = session.sql("""
        SELECT 
            ROUND(avg_quality_score,0) as quality_bucket,
            AVG(overall_conversion_pct) as avg_conversion
        FROM engagement_metrics
        WHERE avg_quality_score IS NOT NULL
        GROUP BY quality_bucket
        ORDER BY quality_bucket
    """).to_pandas()
    
    st.line_chart(quality_perf.set_index('QUALITY_BUCKET')['AVG_CONVERSION'])
    st.caption("Correlation: Higher AI quality scores predict better conversion rates")

# Content recommendations
st.markdown("---")
st.subheader("💡 Optimization Recommendations")

recommendations = session.sql("SELECT * FROM content_recommendations LIMIT 8").to_pandas()
st.dataframe(recommendations, use_container_width=True, hide_index=True)

# Download
st.markdown("---")
csv = ab_results.to_csv(index=False)
st.download_button("📥 Download AB Test Results", data=csv, file_name="ab_test_results.csv", mime="text/csv")

---

##  Tutorial Complete!

### What You've Learned

1.  Used **CORTEX.COMPLETE()** for AI content scoring
2.  Analyzed engagement funnels (impressions → conversions)
3.  Performed A/B testing analysis with SQL
4.  Calculated lift and statistical significance
5.  Generated optimization recommendations
6.  Built content performance dashboards

### Key Cortex Functions

**`AI_COMPLETE(model, prompt)`**
- LLM-powered text generation and analysis
- Score content quality objectively
- Extract insights from text
- Models: mistral-large2, llama3, etc.

### Content Optimization Insights

- **Clarity matters**: Simple, direct messaging performs 30% better
- **Strong CTAs**: Clear calls-to-action improve conversion 2-3x
- **Audience fit**: HCP content needs clinical data, patients need stories
- **Testing wins**: A/B testing reveals 20-40% lift opportunities

### Business Applications

- **Email optimization**: Test subject lines and body content
- **Website testing**: Optimize landing pages and product pages
- **HCP materials**: Score detailing aids for effectiveness
- **Campaign planning**: Predict performance before launch

### Next Steps

- Integrate real campaign data
- Add multivariate testing (3+ variants)
- Build predictive models for content performance
- Create automated content scoring pipelines
- Add competitor content analysis

### Resources

- [Cortex COMPLETE](https://docs.snowflake.com/en/user-guide/snowflake-cortex/llm-functions#label-cortex-llm-complete)
- [Cortex LLM Functions](https://docs.snowflake.com/en/user-guide/snowflake-cortex/llm-functions)

---

**Congratulations!** You've built a complete AI-powered content effectiveness system.

---

## Limpieza del Entorno (Opcional)

### Qué Hace Esto

This cell will **completely remove** all objects created in this tutorial:
- Drops the `CONTENT_DB` database (and all tables within it)
- Drops the `COMPUTE_WH` warehouse

### When to Use

Run this if you want to:
- Clean up your Snowflake environment after completing the tutorial
- Start fresh and re-run the entire notebook
- Remove all demo data

### Instructions

**This cell is commented out by default** to prevent accidental deletion when running "Run All".

To reset your environment:
1. **Remove the comment markers** (`--`) from the SQL commands below
2. **Run this cell manually**

 **Warning**: This action cannot be undone. All data will be permanently deleted.

In [ ]:
-- Descomentar las líneas siguientes and ejecutar esta celda para limpiar el entorno

-- DROP DATABASE IF EXISTS CONTENT_DB;
-- DROP WAREHOUSE IF EXISTS COMPUTE_WH;
